<a href="https://colab.research.google.com/github/whitestones011/deep_learning/blob/colab/prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata

In [154]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [157]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

In [158]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [159]:
model = AutoModelForCausalLM.from_pretrained(model_id)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## Helper

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [268]:
import re

def generate(model, tokenizer, prompt, max_new_tokens=200):
    messages = [
        {"role": "user", "content": prompt},
    ]
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.max_new_tokens = max_new_tokens
    input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt')
    input_length = input_ids.shape[-1]
    output = model.generate(
        input_ids,
        )
    output_without_prompt = output[:, input_length: ]
    response = tokenizer.decode(output_without_prompt[0])
    response = re.sub('\<.*\>\s*', '', response)
    return response

## Principles of prompting


*   Clear and specific instructions
  * use delimeters: """", <>, ---, ```, tags: `<tag></tag>`
*   Ask for structured output like HTML or JSON
*   Check if conditions are satisfied (edge-cases)
*   Few-shot prompting
*   Give a model time to think
  * Specify the steps to complete a task
  * Instruct model to work out its own solution



In [269]:
text = """
Two Nasa-funded US institutions have been granted access to the lunar samples collected by the Chang'e-5 mission in 2020,
the China National Space Administration (CNSA) said on Thursday.
CNSA chief Shan Zhongde said that the samples were "a shared treasure for all humanity," local media reported.
Chinese researchers have not been able to access Nasa's Moon samples because of restrictions
imposed by US lawmakers on the space agency's collaboration with China.
"""

In [217]:
prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```
{text}
```
"""

In [218]:
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

China's space agency, CNSA, has been granted access to lunar samples collected by the US's Chang'e-5 mission after being restricted by US lawmakers.


## Summarize with a word/sentence/character limit

In [270]:
prompt = f"""
Summarize the text delimited by triple backticks, in the most 10 words.

Text: ```{text}```
"""
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

NASA's lunar samples available to China after restrictions lifted.


## Json output

In [235]:
prompt = f"""
Generate a list of three dog bread along with their heigh and grooming requirements.\
Return the response in JSON format with the following keys: bread, height, grooming.\
Do not include any explanations, only provide a  RFC8259 compliant JSON response  following this format without deviation.\n
"""

response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

```json
{
  "breads": [
    {
      "name": "Bread1",
      "height": "20cm",
      "grooming": "Low"
    },
    {
      "name": "Bread2",
      "height": "30cm",
      "grooming": "Medium"
    },
    {
      "name": "Bread3",
      "height": "40cm",
      "grooming": "High"
    }
  ]
}
```


In [259]:
text ="""
When excercising you need first to do stretching for 5 min. After that you can start with cardio.
Finish with 5 min meditation.
"""

prompt = (
f"""
You will be provided with text delimited by triple backticks.
If the text contains a sequence of instructions, re-write those instructions in the following format:

step : do something
step : do something

Return list of the steps.

```{text}```

"""
)
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

```python
steps = [
    "When excercising you need first to do stretching for 5 min.",
    "After that you can start with cardio.",
    "Finish with 5 min meditation."
]
```

steps : When excercising you need first to do stretching for 5 min. 
steps : After that you can start with cardio. 
steps : Finish with 5 min meditation.


In [267]:
# IMPROVED PROMPT WITH STEP BY STEP INSTRUCTIONS

text ="""
When excercising you need first to do stretching for 5 min. After that you can start with cardio.
Finish with 5 min meditation.
"""

prompt = (
f"""
Perform the following actions:
1 - You will be provided with text delimited by <>.
2 - If the text contains a sequence of instructions, create as list of those instructions.
3 - The instruction should be written in one word.
4 - Output a json object that contains the following keys: step, instruction.

Use the following format:
Text: <text with instructions>
Instructions: <list of instructions from text>
Output JSON: <json with step and instruction>

Text: <{text}>

"""
)
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

I can perform the actions for you.

Text: Instructions: 
- exercise
- stretch
- cardio
- meditation

Output JSON:
```json
{
  "step": "Instructions",
  "instruction": "['exercise','stretch', 'cardio','meditation']"
}
```


## Inferring

In [273]:
text = """
I ordered stuff from online shop 2 weeks ago and now got message they cannot make calls to international numbers.
"""

In [275]:
prompt = f"""
What is the sentiment of the following product review, which is delimited with triple backticks?

No explanation needed. Give an answer in a single word, either "positive" or "negative".

Review text: ```{text}``
"""
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

Negative


## Extract data

In [277]:
text = """
I have placed an order for purchase of a gift, number AXFHJSG1. What is the dispatch date?
"""

In [280]:
prompt  = f"""
Identify the followimng items from the text:
- Item purchased
- Order number

Format the response as a JSON object with the following keys: item, order_number.

Text: {text}
"""
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)

{
  "item": "gift",
  "order_number": "AXFHJSG1"
}


## Transforming

In [284]:
text = "What is for diner tonight?"

prompt = f"""
Translate the following English text to Spanish.
No explanation needed.

```{text}```
"""

response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)


¿Qué hay para cenar esta noche?


## Spellchecks

In [291]:
text = "Im workinh all day and will stay late too."

prompt = f"""
Perform the following actions:
1 - Read the text delimited by triple backticks
2 - Proofread the text
3 - Write improved version of the text

```{text}```
"""

response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)


Here are the actions performed on the given text:

1. **Reading the text**: The text is "Im workinh all day and will stay late too."

2. **Proofreading the text**: The text has a few spelling and grammar errors, such as 'Im' instead of 'I'm', 'workinh' instead of 'working', and 'too' instead of 'to'.

3. **Improved version of the text**: 
"I work hard all day and will stay late too. This is a typical day for me."

In this improved version, I've corrected the spelling and grammar errors, and also added a more formal and polite tone to the text.


## Expanding: automatic replies

In [296]:
text = """
I have placed an order for purchase of a gift, number AXFHJSG1. What is the dispatch date?
"""

In [298]:
text

'\nI have placed an order for purchase of a gift, number AXFHJSG1. What is the dispatch date?\n'

In [299]:
sentiment = "negative"

prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
You received the message delimited by triple backticks.
Generate a reply to thank the customer for their message.
If the sentiment is positive or neutral, thank them for their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service.
Make sure to use specific details from the review.
Write in a concise and professional tone.

Customer message: ```{text}```
Sentiment: {sentiment}

"""
response = generate(model, tokenizer, prompt, max_new_tokens=1000)
print(response)


```
Dear valued customer,

Thank you for reaching out to us regarding your order. We appreciate your feedback and concern about the dispatch date for your gift order, number AXFHJSG1. Unfortunately, we were unable to locate any information on a customer with this number.

If you have any further questions or concerns, please don't hesitate to contact us. You can reach out to us via phone at 1-800-GIFT-2020 or email at [customer service email]. We are here to assist you.

Thank you for choosing our company, and we hope to serve you better in the future.

Best regards,
Customer Service Team
```


# Chatbot

In [310]:
def get_completion(model, tokenizer, messages, max_new_tokens=200, temperature=0):
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.max_new_tokens = max_new_tokens
    model.generation_config.temperature = temperature
    input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt')
    input_length = input_ids.shape[-1]
    output = model.generate(
        input_ids,
        )
    output_without_prompt = output[:, input_length: ]
    response = tokenizer.decode(output_without_prompt[0])
    response = re.sub('\<.*\>\s*', '', response)
    return response

In [311]:
messages =  [
  {'role':'system', 'content':'You are a helpfull assistant.'},
  {'role':'user', 'content':'Hi, Im Lise'},
]

In [312]:
response = get_completion(model, tokenizer, messages, max_new_tokens=100, temperature=.1)
print(response)

Hello Lise! How can I assist you today?


In [317]:
messages.append({'role': 'asssitant', 'content': 'Hello Lise! How can I assist you today?'})
messages.append({'role':'user', 'content':'Do you remember my name?'})
response = get_completion(model, tokenizer, messages, max_new_tokens=100, temperature=.1)
print(response)

I'm a large language model, I don't have the ability to remember your name or any other personal information about you. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations.

However, I'm happy to chat with you and get to know you better if you'd like! What's on your mind today, Lise?


# ORDER BOT

In [328]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion(model, tokenizer, context, max_new_tokens=100, temperature=.1)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)


In [320]:
!pip install jupyter_bokeh -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.6 MB/s eta 0:00:00


In [318]:
import panel as pn

In [321]:
pn.extension()

In [332]:
panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
If the pizza size not provided, ask customer what size it needs.\
You respond in a short, very conversational friendly style. \
Make sure you offer options from the menu to the customer.
You respond in a short, very conversational friendly style.
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages

In [ ]:
inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=1000),
)

dashboard

In [ ]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response =  get_completion(model, tokenizer, messages, temperature=.1)
print(response)